In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import os
# df = pd.read_csv(os.path.join('..', 'data', 'data_clean.csv'), index_col=0).reset_index()
df = pd.read_csv(os.path.join('..', 'data', 'data_clean.csv'), index_col=0)
df

,Residence_type,gender,hypertension,heart_disease,ever_married,work_type,smoking_status,age,avg_glucose_level,bmi,stroke
0,Urban,Male,0,1,Yes,Private,formerly smoked,67.0,228.69,36.6,1
2,Rural,Male,0,1,Yes,Private,never smoked,80.0,105.92,32.5,1
3,Urban,Female,0,0,Yes,Private,smokes,49.0,171.23,34.4,1
4,Rural,Female,1,0,Yes,Self-employed,never smoked,79.0,174.12,24.0,1
5,Urban,Male,0,0,Yes,Private,formerly smoked,81.0,186.21,29.0,1
...,...,...,...,...,...,...,...,...,...,...,...
5104,Rural,Female,0,0,No,children,Unknown,13.0,103.08,18.6,0
5106,Urban,Female,0,0,Yes,Self-employed,never smoked,81.0,125.20,40.0,0
5107,Rural,Female,0,0,Yes,Self-employed,never smoked,35.0,82.99,30.6,0
5108,Rural,Male,0,0,Yes,Private,formerly smoked,51.0,166.29,25.6,0


### Feature Engineering and Transformations

In [3]:
df['is_urban'] = (df['Residence_type'] == 'Urban').astype(np.int8)
df['is_male'] = (df['gender'] == 'Male').astype(np.int8)
df['ever_married'] = (df['ever_married'] == 'Yes').astype(np.int8)
df.head()

,Residence_type,gender,hypertension,heart_disease,ever_married,work_type,smoking_status,age,avg_glucose_level,bmi,stroke,is_urban,is_male
0,Urban,Male,0,1,1,Private,formerly smoked,67.0,228.69,36.6,1,1,1
2,Rural,Male,0,1,1,Private,never smoked,80.0,105.92,32.5,1,0,1
3,Urban,Female,0,0,1,Private,smokes,49.0,171.23,34.4,1,1,0
4,Rural,Female,1,0,1,Self-employed,never smoked,79.0,174.12,24.0,1,0,0
5,Urban,Male,0,0,1,Private,formerly smoked,81.0,186.21,29.0,1,1,1


In [4]:
# df = df[[
#     'age', 'is_male', 'is_urban', 'ever_married', 'work_type', 'smoking_status',
#     'bmi', 'hypertension', 'heart_disease', 'avg_glucose_level', 'stroke'
# ]]
# df.head()

In [5]:
df.dtypes

Residence_type        object
gender                object
hypertension           int64
heart_disease          int64
ever_married            int8
work_type             object
smoking_status        object
age                  float64
avg_glucose_level    float64
bmi                  float64
stroke                 int64
is_urban                int8
is_male                 int8
dtype: object

Things like age groups and bmi categories can help the model pick up on non-linear patterns that raw continuous values sometimes miss.

In [6]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 35, 50, 65, np.inf],
    labels=['<18', '18-35', '36-50', '51-65', '65+']
)

df['bmi_category'] = pd.cut(
    df['bmi'],
    bins=[0, 18.5, 25, 30, np.inf],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)

df['avg_glucose_category'] = pd.cut(
    df['avg_glucose_level'],
    bins=[0, 70, 100, 250, np.inf],
    labels=['Low', 'Normal', 'High', 'VeryHigh']
)

In [7]:
df['ever_smoked'] = (df['smoking_status'].apply(
    lambda x: 1 if x in ['formerly smoked', 'smokes'] else 0
)).astype(np.int8)

df['age_bmi_interaction'] = df['age'] * df['bmi']

df['health_risk_score'] = (
    df['hypertension'] + df['heart_disease'] + df['ever_smoked'] + df['age'] / 100
)

df['work_residence_interaction'] = df['work_type'] + "_" + df['Residence_type']

df['marital_age_interaction'] = df['ever_married'] * df['age']

df.sample(3)

,Residence_type,gender,hypertension,heart_disease,ever_married,work_type,smoking_status,age,avg_glucose_level,bmi,...,is_urban,is_male,age_group,bmi_category,avg_glucose_category,ever_smoked,age_bmi_interaction,health_risk_score,work_residence_interaction,marital_age_interaction
3158,Rural,Female,0,0,0,Private,formerly smoked,16.0,95.38,34.3,...,0,0,<18,Obese,Normal,1,548.8,1.16,Private_Rural,0.0
2260,Urban,Female,0,0,1,Private,Unknown,47.0,96.04,29.2,...,1,0,36-50,Overweight,Normal,0,1372.4,0.47,Private_Urban,47.0
1604,Urban,Female,0,0,1,Private,never smoked,47.0,65.04,30.9,...,1,0,36-50,Obese,Low,0,1452.3,0.47,Private_Urban,47.0


In [8]:
df.dtypes

Residence_type                  object
gender                          object
hypertension                     int64
heart_disease                    int64
ever_married                      int8
work_type                       object
smoking_status                  object
age                            float64
avg_glucose_level              float64
bmi                            float64
stroke                           int64
is_urban                          int8
is_male                           int8
age_group                     category
bmi_category                  category
avg_glucose_category          category
ever_smoked                       int8
age_bmi_interaction            float64
health_risk_score              float64
work_residence_interaction      object
marital_age_interaction        float64
dtype: object